# 🚀 Bloque 7: Despliegue de Modelos en Producción

**Objetivo:** Pasar un modelo de notebook a un endpoint de API real, contenerizado y con tracking de experimentos.

---

## 1. ¿Qué significa "desplegar un modelo"?

Entrenar un modelo en Jupyter es solo el 20% del trabajo. El 80% restante es hacerlo accesible y confiable en producción:

```
Notebook (investigación)
    ↓
Serialización del modelo (guardar pesos)
    ↓
API REST con FastAPI (exponer el modelo)
    ↓
Docker (contenerizar para reproducibilidad)
    ↓
MLflow (tracking de versiones y experimentos)
    ↓
Producción (cloud, on-premise, HPC)
```

---

## 2. Serialización del modelo

In [ ]:
import torch
import torch.nn as nn
import pickle
import os

os.makedirs('saved_models', exist_ok=True)

# Modelo de ejemplo
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(),
            nn.Linear(32, 3)
        )
    def forward(self, x):
        return self.net(x)

model = SimpleModel()

# --- Opción 1: Guardar solo los pesos (recomendado para PyTorch) ---
torch.save(model.state_dict(), 'saved_models/model_weights.pt')

# Cargar
model_loaded = SimpleModel()
model_loaded.load_state_dict(torch.load('saved_models/model_weights.pt'))
model_loaded.eval()
print("✅ Modelo cargado desde state_dict")

# --- Opción 2: TorchScript (para producción sin Python) ---
scripted_model = torch.jit.script(model)
scripted_model.save('saved_models/model_scripted.pt')
print("✅ Modelo guardado como TorchScript")

# --- Opción 3: Pickle (para sklearn) ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

iris = load_iris()
rf = RandomForestClassifier(n_estimators=10)
rf.fit(iris.data, iris.target)

with open('saved_models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

with open('saved_models/rf_model.pkl', 'rb') as f:
    rf_loaded = pickle.load(f)

print("✅ Modelo sklearn guardado y cargado con pickle")

## 3. API con FastAPI

FastAPI es el framework más moderno y rápido para crear APIs en Python. Genera documentación automática.

### Estructura de una API de ML:

In [ ]:
# Escribe este código en un fichero 'app.py' y ejecuta: uvicorn app:app --reload

api_code = '''
# app.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Iris Classifier API", version="1.0")

# Cargar modelo al inicio (no en cada request)
with open("saved_models/rf_model.pkl", "rb") as f:
    model = pickle.load(f)

classes = ["setosa", "versicolor", "virginica"]

# Schema de entrada con validación automática
class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/")
def root():
    return {"message": "Iris Classifier API running"}

@app.post("/predict")
def predict(features: IrisFeatures):
    X = np.array([[features.sepal_length, features.sepal_width,
                   features.petal_length, features.petal_width]])
    
    prediction = model.predict(X)[0]
    probabilities = model.predict_proba(X)[0]
    
    return {
        "class": classes[prediction],
        "class_id": int(prediction),
        "probabilities": {
            cls: float(prob)
            for cls, prob in zip(classes, probabilities)
        }
    }

@app.get("/health")
def health():
    return {"status": "healthy"}
'''

with open('app.py', 'w') as f:
    f.write(api_code)

print("✅ app.py creado")
print("\nPara lanzar la API:")
print("  pip install fastapi uvicorn")
print("  uvicorn app:app --reload --host 0.0.0.0 --port 8000")
print("\nDocumentación automática en: http://localhost:8000/docs")

In [ ]:
# Probar la API con requests
# (ejecuta esto solo si tienes la API corriendo)

test_code = '''
import requests

url = "http://localhost:8000/predict"
payload = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = requests.post(url, json=payload)
print(response.json())
# Output esperado: {"class": "setosa", "class_id": 0, "probabilities": {...}}
'''
print("Código para probar la API:")
print(test_code)

## 4. Dockerfile — contenerizar el modelo

In [ ]:
dockerfile = '''
# Dockerfile
FROM python:3.10-slim

WORKDIR /app

# Instalar dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar código y modelo
COPY app.py .
COPY saved_models/ ./saved_models/

# Exponer puerto
EXPOSE 8000

# Comando de arranque
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements = '''
fastapi==0.104.0
uvicorn==0.24.0
scikit-learn==1.3.0
numpy==1.24.0
pydantic==2.4.0
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("✅ Dockerfile y requirements.txt creados")
print("\nComandos Docker:")
print("  docker build -t iris-api .")
print("  docker run -p 8000:8000 iris-api")
print("  docker ps  # Ver contenedores activos")

## 5. MLflow — tracking de experimentos

In [ ]:
# !pip install mlflow

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.datasets import load_iris

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Configurar MLflow
mlflow.set_experiment("iris_classification")

# Probar diferentes configuraciones
configs = [
    {'n_estimators': 10,  'max_depth': 3},
    {'n_estimators': 50,  'max_depth': 5},
    {'n_estimators': 100, 'max_depth': None},
]

for config in configs:
    with mlflow.start_run():
        # Registrar parámetros
        mlflow.log_params(config)
        
        # Entrenar
        rf = RandomForestClassifier(**config, random_state=42)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        
        # Registrar métricas
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='macro')
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('f1_macro', f1)
        
        # Registrar modelo
        mlflow.sklearn.log_model(rf, 'model')
        
        print(f"Config: {config} → Accuracy: {acc:.3f}, F1: {f1:.3f}")

print("\n✅ Experimentos registrados en MLflow")
print("Ver UI: mlflow ui --host 0.0.0.0 --port 5000")

## 6. Resumen del ciclo completo

```
1. DATOS        → pandas, numpy, Feature Engineering
2. MODELADO     → scikit-learn / PyTorch / HuggingFace / Pyro
3. EVALUACIÓN   → MLflow tracking de métricas y parámetros  
4. SERIALIZAR   → pickle / torch.save / TorchScript
5. API          → FastAPI + Pydantic + uvicorn
6. CONTENER     → Docker (Dockerfile + requirements.txt)
7. DESPLEGAR    → Cloud (Azure, GCP, AWS) / on-premise / HPC
```

---

## ✅ Resumen del bloque

- Serializas modelos con **state_dict, pickle y TorchScript**
- Creas una **API REST** con FastAPI lista para producción
- Contenerizas la API con **Docker**
- Registras y comparas experimentos con **MLflow**
- Entiendes el **ciclo completo** de un modelo en producción

---

## 🎉 ¡Has completado todos los bloques!

Con estos 7 bloques tienes la base completa para el rol de **Python Developer (IA)**.

**Próximos pasos recomendados:**
1. Completa el [Fast.ai Course](https://course.fast.ai) para profundizar en DL
2. Haz el [HuggingFace Course](https://huggingface.co/learn) completo
3. Construye un proyecto end-to-end propio y ponlo en este repo
4. Estudia Pyro con los [tutoriales oficiales](https://pyro.ai/examples/)